# 图统计可视化

本笔记本可视化若干真实数据集和合成数据集的图统计量。我们观察到，`src.core.combined_syn` 生成的合成数据覆盖了真实图中许多基础统计特征的范围。

首先我们比较所有图数据集的聚类系数和最短路径长度。可以看到，合成数据（`syn`）覆盖了 `cox2` 和 `reddit-binary` 所覆盖的区域。由于蛋白质结构的原因，ENZYMES 包含一些更长、更细的图，因此在分布上也更有特点。

In [1]:
import sys
from pathlib import Path

repo_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repository root")
repo_root = str(repo_root)
sys.path.insert(0, repo_root)

import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import pickle
from src.core import data, utils, combined_syn

In [ ]:
# 对比聚类系数与最短路径长度
for dataset in ["enzymes", "cox2", "reddit-binary", "syn"]:
    if dataset == "syn":
        gen = combined_syn.get_generator([11])
        graphs = [gen.generate() for i in range(10000)]
    else:
        train, test, task = data.load_dataset(dataset)
        graphs = []
        for i in range(10000):
            graph, neigh = utils.sample_neigh(train, 11)
            graphs.append(graph.subgraph(neigh))

    clustering = [nx.average_clustering(G.subgraph(max(nx.connected_components(G), key=len))) for G in graphs]
    path_length = [nx.average_shortest_path_length(G.subgraph(max(nx.connected_components(G), key=len))) for G in graphs]
    plt.scatter(clustering, path_length, s=10, label=dataset)

plt.xlabel("Clustering")
plt.ylabel("Shortest path length")
plt.legend()
plt.show()

接下来我们比较不同图类型的直径和密度，并再次看到合成数据通常能够覆盖真实世界图的主要范围。

In [ ]:
# 对比直径与密度
for dataset in ["enzymes", "cox2", "reddit-binary", "syn"]:
    if dataset == "syn":
        gen = combined_syn.get_generator([11])
        graphs = [gen.generate() for i in range(10000)]
    else:
        train, test, task = data.load_dataset(dataset)
        graphs = []
        for i in range(10000):
            graph, neigh = utils.sample_neigh(train, 11)
            graphs.append(graph.subgraph(neigh))

    diameter = [nx.diameter(G.subgraph(max(nx.connected_components(G), key=len))) for G in graphs]
    density = [nx.density(G.subgraph(max(nx.connected_components(G), key=len))) for G in graphs]
    plt.scatter(diameter, density, s=10, label=dataset)

plt.xlabel("Diameter")
plt.ylabel("Density")
plt.legend()
plt.show()